# Exercise: Debugging Neural Network Training

Use this notebook to reproduce common training failures and practice fixing them systematically.

Suggested workflow:

1. Choose one issue from the exercise menu.
2. Run the bad baseline for that issue and inspect the learning curves.
3. Change one intervention at a time.
4. Record what changed in train loss, validation loss, F1, and gradient norms.

Rule of thumb: do not tune on the test set. Use the validation split for decisions and keep the test split for the final check.

In [ ]:
import plotly.express as px

from exercise_3_training_helpers import run_experiment, get_symptoms_for_issue, get_fixes_for_issue

## Tunable Training Parameters

Use `custom_overrides` to change experiment behavior in a controlled, one-change-at-a-time workflow.

#### Model architecture
- `input_size`: `train_x_tensor.shape[1]`  
    Number of input features.
- `hidden_dims`: `[64, 32]`  
    Hidden layer sizes.
- `activation`: `"relu"`  
    Hidden-layer activation function.
- `dropout`: `0.1`  
    Dropout probability for regularization.
- `use_batch_norm`: `False`  
    Enables/disables batch normalization.
- `output_activation`: `None`  
    Output-layer activation (if needed).
- `init`: `"xavier_uniform"`  
    Weight initialization strategy.

#### Optimization and training
- `learning_rate`: `1e-3`  
    Optimizer step size (often the first thing to tune).
- `optimizer_name`: `"adam"`  
    Optimizer type.
- `weight_decay`: `1e-4`  
    L2 regularization strength.
- `gradient_clip_norm`: `None`  
    Gradient clipping threshold to prevent exploding gradients.
- `epochs`: `10`  
    Number of training epochs.
- `batch_size`: `512`  
    Mini-batch size.

#### Class balance and feature processing
- `balance_strategy`: `"none"`  
    Class imbalance handling strategy.
- `use_scaled_features`: `True`  
    Whether to standardize/scale input features.
- `subset_fraction`: `1.0`  
    Fraction of training data to use.

#### Data corruption / robustness controls
- `label_noise`: `0.0`  
    Fraction of labels to corrupt.
- `train_feature_noise`: `0.0`  
    Noise level added to training features.
- `rare_feature_fraction`: `0.0`  
    Fraction of rare/high-impact features.
- `rare_feature_scale`: `25.0`  
    Scale factor for rare features.
- `distribution_shift_strength`: `0.0`  
    Train/validation distribution shift intensity.
- `leak_validation_into_train`: `False`  
    Simulates data leakage (debugging only).

#### Reproducibility
- `seed`: `SEED`  
    Random seed for deterministic comparisons.

---

### Practical tuning guideline
For reliable debugging, modify **one parameter at a time**, then compare:
- validation `loss`
- validation `f1`
- `mean_grad_norm`
- `positive_prediction_rate`

Keep only changes that improve validation behavior before trying a second adjustment.

The performance of the basic model, try to get to this result

In [ ]:
_, _, _, test_metrics = run_experiment(None)
display(test_metrics)

## Goal: Systematic Debugging With Minimal `custom_override` Changes

Goal: for each issue, start from the bad baseline and recover validation **loss/F1** while changing as
few settings as possible (ideally 1, max 2 per trial).

### Core workflow (repeat for each `selected_issue`)

1. Run baseline (`run_experiment(selected_issue)`).
2. Apply **one** targeted override.
3. Re-run and compare:
    - `loss` (validation)
    - `f1`
    - `mean_grad_norm`
    - `positive_prediction_rate`
4. Keep only changes that improve validation behavior.
5. Only after stabilizing, test a second small change.
6. Do **not** tune on test metrics.

---

## What your current run suggests (issue 0)

Observed pattern: very high/erratic train loss, large gradient spikes, F1 = 0, and zero positive
predictions.  
Most likely first fix: lower LR before anything else.

Recommended sequence:
1. `{"learning_rate": 1e-3}`
2. If still weak: `{"activation": "leaky_relu"}`
3. If spikes persist: `{"gradient_clip_norm": 1.0}`

This keeps interventions minimal and makes causal impact easier to interpret.

In [ ]:
selected_issue = 0

overwrite = {}
_, trained_model, training_history, test_metrics = run_experiment(selected_issue, custom_overrides=overwrite)

plot_df = training_history.melt(
    id_vars="epoch",
    value_vars=["train_loss", "loss", "f1", "mean_grad_norm"],
    var_name="metric",
    value_name="value",
)

fig = px.line(
    plot_df,
    x="epoch",
    y="value",
    color="metric",
    markers=True,
    title="Training Metrics by Epoch",
)

fig.update_layout(
    yaxis_title="metric value",
    xaxis_title="epoch",
    template="plotly_white",
)

fig.show()

display(test_metrics)
display(trained_model)

In case you don't know what the issue is you can also get a description of the sympthons which

In [ ]:
get_symptoms_for_issue(selected_issue)

Or even some ideas on how to solve it

In [ ]:
get_fixes_for_issue(selected_issue)